# 10 — Completion by Task Type and Backlog Age

These reports describe recorded task starts, not health or productivity. Changes may reflect seasons, travel, or recording habits. Filters use exact names; `None` includes all values. Choose a task type to focus rolling and seasonal comparisons on one activity. Missing records are not proof of inactivity. Engagement-domain reports (including brief section 5) are deferred with the section 3 schema enhancement.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))
from support.activity_analysis import (
    ReportPeriod,
    backlog_by_age,
    completion_by_type,
    filter_history,
)

In [ ]:
# Inclusive dates; END_DATE is also the as-of date. Use finite ISO dates.
START_DATE = "2024-01-01"
END_DATE = date.today().isoformat()
CATEGORY = None
TASK_TYPE = None
LOCATION = None
ROLLING_WINDOW = 90
DORMANCY_THRESHOLD = 90

In [ ]:
period = ReportPeriod(START_DATE, END_DATE, CATEGORY, TASK_TYPE, LOCATION)
# Load earlier observations for rolling windows, continuity and the full backlog.
query = construct_query("task-history.sql", {"START-DATE": "0001-01-01", "END-DATE": END_DATE})
history = filter_history(query_data(query), period)
filters = f"category={CATEGORY or 'All'}, type={TASK_TYPE or 'All'}, location={LOCATION or 'All'}"
period_label = f"{START_DATE} to {END_DATE}\n{filters}"
print(period_label)

## Completion time by task type

Elapsed time is in calendar days, including same-day completions. Statistics describe tasks started in the period and completed by its end; unfinished tasks are excluded, not assigned zero. The 90th percentile uses linear interpolation.


In [ ]:
completion = completion_by_type(history, period)
completion

In [ ]:
if not completion.empty and len(completion.columns) > 1:
    figure, axes = plt.subplots(
        figsize=(CHART_FIGURE_SIZE[0], max(CHART_FIGURE_SIZE[1], len(completion) * 0.3))
    )
    completion.assign(Activity=completion["Category"] + " / " + completion["Task Type"]).plot(
        x="Activity", y=["90th Percentile Days"], kind="barh", stacked=False, ax=axes
    )
    format_chart(
        figure,
        axes,
        f"90th percentile completion time — {period_label}",
        "Days",
        "Activity",
        grid_axis="x",
    )
    export_chart(get_export_folder_path(), "completion-by-type", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Open tasks by category and age

The backlog includes all selected tasks started on or before END_DATE that were not completed by that date, including starts before START_DATE. It reconstructs status from current dates and cannot recover deleted tasks or historical edits.


In [ ]:
backlog = backlog_by_age(history, period)
backlog_types = backlog_by_age(history, period, by_type=True)
display(backlog, backlog_types)

In [ ]:
if not backlog.empty and len(backlog.columns) > 1:
    figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
    backlog.plot(x="Category", y=None, kind="bar", stacked=True, ax=axes)
    format_chart(
        figure,
        axes,
        f"Open tasks at {END_DATE} — {period_label}",
        "Category",
        "Tasks",
        grid_axis="y",
    )
    export_chart(get_export_folder_path(), "backlog-category-age", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Export

Tables are exported as XLSX and charts as PNG to `data/reports/` (or the runner’s configured destination).


In [ ]:
export_to_spreadsheet(
    get_export_folder_path(),
    "completion-by-type-and-backlog.xlsx",
    {
        "Completion by Type": completion,
        "Backlog by Category": backlog,
        "Backlog by Type": backlog_types,
    },
)